#**Pneumonia Detection — Cross-Hospital Distribution Shift**
##**Training Strategy Evaluation with MobileNetV2**

This notebook evaluates three training strategies for pneumonia detection on chest X-rays under cross-hospital distribution shift.

**Source domain:** NIH ChestX-ray14 (training, validation, in-domain test)  
**Target domain:** Guangzhou Chest X-ray (out-domain test only)  
**Backbone     :** MobileNetV2 (ImageNet pretrained, fixed across all strategies)

**Strategies compared:**
1. Frozen Baseline — only the classification head trains
2. Partial Fine-Tuning — last 30 backbone layers + head train
3. Full Fine-Tuning + Class Weighting + Light Augmentation — entire model trains

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#**Experiment**

##**Setup and Configuration**

Mount Google Drive

In [ ]:
import shutil, os

if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("✓ Cleared leftover /content/drive directory")

from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted at /content/drive")

### 1. Environment Setup

Install dependencies, set reproducibility seeds, verify GPU availability.

Install packages

In [ ]:
!pip install -q datasets
print("Packages installed.")

 Imports and seeds

In [ ]:
import os, random, json, time, gc, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"  TensorFlow version : {tf.__version__}")
print(f"  GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")
print(f"  Random seed        : {SEED}")
print("=" * 60)

### 2. Configuration

All paths and hyperparameters are defined here in one place. Modify only this cell when re-running with different settings.

Paths and hyperparameters

In [ ]:
# ---------- Dataset paths ----------
NIH_CSV       = "/content/drive/MyDrive/SEMESTER EMPAT/Research Methodology/Dataset/NIH-Chest-X-ray-dataset/data/Data_Entry_2017_v2020.csv"
NIH_ZIP_DIR   = "/content/drive/MyDrive/SEMESTER EMPAT/Research Methodology/Dataset/NIH-Chest-X-ray-dataset/data/images"
NIH_IMG_LOCAL = "/content/nih_images/images"

# ---------- Output folder ----------
OUT_DIR = "/content/drive/MyDrive/SEMESTER EMPAT/Research Methodology/experiment_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Hyperparameters ----------
IMG_SIZE             = 224
BATCH_SIZE           = 32
MAX_EPOCHS           = 30
ES_PATIENCE          = 5
LR_PLATEAU_PATIENCE  = 3
LR_PLATEAU_FACTOR    = 0.5
DROPOUT              = 0.3
DENSE_UNITS          = 128
THRESHOLD            = 0.5

LR_S1 = 1e-4   # frozen baseline
LR_S2 = 1e-5   # partial fine-tuning
LR_S3 = 1e-5   # full fine-tuning

S2_UNFREEZE_LAST = 30
N_NEG_SUBSET     = 20000

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  Image size         : {IMG_SIZE} x {IMG_SIZE}")
print(f"  Batch size         : {BATCH_SIZE}")
print(f"  Max epochs         : {MAX_EPOCHS}")
print(f"  Early stop patience: {ES_PATIENCE}")
print(f"  Threshold          : {THRESHOLD}")
print(f"  LR Strategy 1      : {LR_S1}")
print(f"  LR Strategy 2      : {LR_S2}")
print(f"  LR Strategy 3      : {LR_S3}")
print(f"  Dense units        : {DENSE_UNITS}")
print(f"  Dropout            : {DROPOUT}")
print(f"  S2 unfrozen layers : last {S2_UNFREEZE_LAST}")
print(f"  Output directory   : {OUT_DIR}")
print("=" * 60)

### 3. Extract NIH Images

NIH ChestX-ray14 ships as 12 ZIP files on Drive. Extract them once to Colab's local SSD for fast access. This step is skipped automatically if the images are already extracted.

Extract NIH ZIPs

In [ ]:
def extract_nih_zips():
    if os.path.exists(NIH_IMG_LOCAL) and len(os.listdir(NIH_IMG_LOCAL)) > 100000:
        print("=" * 60)
        print("EXTRACTION SKIPPED — images already present")
        print(f"  Path  : {NIH_IMG_LOCAL}")
        print(f"  Count : {len(os.listdir(NIH_IMG_LOCAL)):,} files")
        print("=" * 60)
        return

    os.makedirs("/content/nih_images", exist_ok=True)
    zips = sorted([f for f in os.listdir(NIH_ZIP_DIR) if f.endswith(".zip")])

    print("=" * 60)
    print(f"EXTRACTING {len(zips)} ZIP FILES")
    print("=" * 60)

    t0 = time.time()
    for i, z in enumerate(zips, 1):
        zp = os.path.join(NIH_ZIP_DIR, z)
        size_mb = os.path.getsize(zp) / 1e6
        t_zip = time.time()
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall("/content/nih_images")
        print(f"  [{i:2d}/{len(zips)}] {z}  ({size_mb:6.0f} MB)  done in {time.time()-t_zip:5.1f}s")

    # Remove macOS junk
    junk = "/content/nih_images/__MACOSX"
    if os.path.exists(junk):
        shutil.rmtree(junk)

    total = len(os.listdir(NIH_IMG_LOCAL))
    print("=" * 60)
    print(f"  Total time : {(time.time()-t0)/60:.1f} min")
    print(f"  Total PNGs : {total:,}")
    print("=" * 60)

extract_nih_zips()

##**NIH and GuangZhou Domain Preparation**

### 4. Load NIH Metadata and Harmonize Labels

Apply the binary labeling rule: any image whose `Finding Labels` contains the substring `"Pneumonia"` is labeled `1`; everything else is labeled `0`. Files that cannot be located on disk are dropped.

Load NIH metadata + label harmonization

In [ ]:
df = pd.read_csv(NIH_CSV)
df["label"]    = df["Finding Labels"].apply(lambda s: 1 if "Pneumonia" in str(s) else 0)
df["filepath"] = df["Image Index"].apply(lambda f: os.path.join(NIH_IMG_LOCAL, f))

n_before = len(df)
df["exists"] = df["filepath"].apply(os.path.exists)
n_missing = (~df["exists"]).sum()
df = df[df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

n_pos = int(df["label"].sum())
n_neg = len(df) - n_pos

print("=" * 60)
print("NIH METADATA — FULL DATASET")
print("=" * 60)
print(f"  Rows in CSV          : {n_before:,}")
print(f"  Missing image files  : {n_missing:,}")
print(f"  Usable rows          : {len(df):,}")
print(f"  Pneumonia (label=1)  : {n_pos:,}  ({n_pos/len(df)*100:.2f}%)")
print(f"  Non-pneumonia (0)    : {n_neg:,}  ({n_neg/len(df)*100:.2f}%)")
print("=" * 60)

### 5. Stratified Subsampling

For Colab feasibility, we keep all pneumonia-positive cases and randomly sample 20,000 non-pneumonia cases. The seed ensures reproducibility.

Subsample

In [ ]:
df_pos = df[df["label"] == 1].copy()
df_neg = df[df["label"] == 0].sample(n=N_NEG_SUBSET, random_state=SEED).copy()
df_sub = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("=" * 60)
print("NIH SUBSET (used for experiment)")
print("=" * 60)
print(f"  Pneumonia (label=1)  : {(df_sub['label']==1).sum():,}")
print(f"  Non-pneumonia (0)    : {(df_sub['label']==0).sum():,}")
print(f"  Total                : {len(df_sub):,}")
print(f"  Pneumonia rate       : {df_sub['label'].mean()*100:.2f}%")
print("=" * 60)

### 6. Patient-Level Stratified Split

Split into 70% train / 15% validation / 15% in-domain test, grouped by `Patient ID` so no patient appears in more than one split. This prevents leakage from patients with multiple X-rays.

Split

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df_sub, df_sub["label"], groups=df_sub["Patient ID"]))
df_train = df_sub.iloc[train_idx].reset_index(drop=True)
df_temp  = df_sub.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(df_temp, df_temp["label"], groups=df_temp["Patient ID"]))
df_val  = df_temp.iloc[val_idx].reset_index(drop=True)
df_test = df_temp.iloc[test_idx].reset_index(drop=True)

assert set(df_train["Patient ID"]) & set(df_val["Patient ID"])  == set()
assert set(df_train["Patient ID"]) & set(df_test["Patient ID"]) == set()
assert set(df_val["Patient ID"])   & set(df_test["Patient ID"]) == set()

print("=" * 78)
print(f"{'NIH SPLITS — Patient-level stratified':^78}")
print("=" * 78)
print(f"{'Split':<10}{'Total':>10}{'Non-pneum.':>14}{'Pneum.':>10}{'Pos %':>10}{'Patients':>14}")
print("-" * 78)
for name, d in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    n = len(d); pos = int(d["label"].sum()); neg = n - pos
    pat = d["Patient ID"].nunique()
    print(f"{name:<10}{n:>10,}{neg:>14,}{pos:>10,}{pos/n*100:>9.2f}%{pat:>14,}")
print("=" * 78)
print("  ✓ No patient overlap between splits")
print("=" * 78)

### 7. Load Guangzhou External Test Set

Loaded from Hugging Face (`hf-vision/chest-xray-pneumonia`), test split only. This dataset is **never used for training, validation, threshold tuning, or model selection**.

Load Guangzhou

In [ ]:
from datasets import load_dataset

ds_gz = load_dataset("hf-vision/chest-xray-pneumonia", split="test")
gz_labels = np.array([int(x["label"]) for x in ds_gz])

n = len(gz_labels); pos = int((gz_labels == 1).sum()); neg = n - pos

print("=" * 60)
print("GUANGZHOU EXTERNAL TEST SET (out-domain only)")
print("=" * 60)
print(f"  Total                : {n:,}")
print(f"  Non-pneumonia (0)    : {neg:,}  ({neg/n*100:.2f}%)")
print(f"  Pneumonia (1)        : {pos:,}  ({pos/n*100:.2f}%)")
print("=" * 60)

##**Data Pipeline, Preprocessing, Model Builder and Training Utilities**

### 8. Build Data Pipelines

NIH and Guangzhou pipelines are matched: both decode → 3-channel → resize bilinear → float32 [0, 255] → preprocess_input → [-1, 1].

Light augmentation is applied **only** to the Strategy 3 NIH training set, **before** `preprocess_input` (the correct pixel range for Keras augmentation layers). No horizontal flip is used because chest X-ray laterality is clinically meaningful.

NIH tf.data pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

def apply_preprocess(img, label):
    return preprocess_input(img), label

augmenter = tf.keras.Sequential([
    layers.RandomRotation(0.05, fill_mode="constant", fill_value=0.0),
    layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0),
    layers.RandomZoom(0.05, fill_mode="constant", fill_value=0.0),
    layers.RandomContrast(0.1),
], name="augmenter")

def apply_augment(img, label):
    img = augmenter(img, training=True)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_nih_ds(df_split, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df_split["filepath"].values, df_split["label"].values.astype(np.float32))
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df_split), 4096),
                        seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(apply_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(apply_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds_plain   = make_nih_ds(df_train, augment=False, shuffle=True)
train_ds_augment = make_nih_ds(df_train, augment=True,  shuffle=True)
val_ds   = make_nih_ds(df_val,   augment=False, shuffle=False)
test_ds  = make_nih_ds(df_test,  augment=False, shuffle=False)

# Sanity check
for x, y in train_ds_plain.take(1):
    pmin, pmax = float(tf.reduce_min(x)), float(tf.reduce_max(x))

print("=" * 60)
print("NIH DATA PIPELINE — sanity check")
print("=" * 60)
print(f"  Batch shape          : {tuple(x.shape)}")
print(f"  Dtype                : {x.dtype.name}")
print(f"  Pixel range          : [{pmin:.3f}, {pmax:.3f}]  (expect ≈ [-1, 1])")
print("=" * 60)

 Guangzhou tf.data pipeline

In [ ]:
def preprocess_pil_to_array(pil_img):
    pil_rgb = pil_img.convert("RGB")
    arr = np.array(pil_rgb, dtype=np.uint8)
    arr = tf.image.resize(arr, [IMG_SIZE, IMG_SIZE]).numpy().astype(np.float32)
    arr = preprocess_input(arr)
    return arr

gz_cache = os.path.join(OUT_DIR, "guangzhou_preprocessed.npz")

if os.path.exists(gz_cache):
    data = np.load(gz_cache)
    gz_X, gz_y = data["X"], data["y"]
    cached = True
else:
    gz_X = np.stack([preprocess_pil_to_array(x["image"]) for x in ds_gz]).astype(np.float32)
    gz_y = gz_labels.astype(np.float32)
    np.savez_compressed(gz_cache, X=gz_X, y=gz_y)
    cached = False

gz_ds = tf.data.Dataset.from_tensor_slices((gz_X, gz_y)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("=" * 60)
print("GUANGZHOU DATA PIPELINE")
print("=" * 60)
print(f"  Source               : {'cache' if cached else 'fresh preprocess + cache'}")
print(f"  Shape                : {gz_X.shape}")
print(f"  Dtype                : {gz_X.dtype}")
print(f"  Pixel range          : [{gz_X.min():.3f}, {gz_X.max():.3f}]  (expect ≈ [-1, 1])")
print("=" * 60)

### 9. MobileNetV2 Model Builder

The same MobileNetV2 backbone is used for all three strategies. Only the freezing pattern and learning rate differ.

| Strategy | Backbone state | Learning rate |
|---|---|---|
| 1. Frozen Baseline | All frozen | 1e-4 |
| 2. Partial Fine-Tuning | Last 30 layers unfrozen | 1e-5 |
| 3. Full FT + CW + Aug | All unfrozen | 1e-5 |

The classification head is identical across strategies: `GlobalAveragePooling2D → Dropout(0.3) → Dense(128, ReLU) → Dense(1, Sigmoid)`.

Model builder

In [ ]:
def build_model(strategy):
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                       include_top=False, weights="imagenet")

    if strategy == 1:
        base.trainable = False
    elif strategy == 2:
        base.trainable = True
        for layer in base.layers[:-S2_UNFREEZE_LAST]:
            layer.trainable = False
    elif strategy == 3:
        base.trainable = True
    else:
        raise ValueError("strategy must be 1, 2, or 3")

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False if strategy == 1 else None)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(DENSE_UNITS, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs, name=f"mobilenetv2_strategy{strategy}")

    lr = {1: LR_S1, 2: LR_S2, 3: LR_S3}[strategy]
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
    total     = sum(int(np.prod(v.shape)) for v in model.weights)

    print("-" * 60)
    print(f"  Model              : Strategy {strategy}")
    print(f"  Total params       : {total:,}")
    print(f"  Trainable params   : {trainable:,}  ({trainable/total*100:.1f}%)")
    print(f"  Learning rate      : {lr}")
    print("-" * 60)
    return model

### 10. Training Function with Auto-Resume

If a strategy's model has been saved already, it is loaded from disk instead of retraining. This protects against Colab disconnects — re-running a training cell after reconnecting just reloads the model in seconds.

To force retraining of a specific strategy, pass `force_retrain=True` in its training cell.

Training helper

In [ ]:
def train_strategy(strategy, train_dataset, class_weight=None, tag="", force_retrain=False):
    save_path    = os.path.join(OUT_DIR, f"strategy{strategy}_model.h5")
    history_path = os.path.join(OUT_DIR, f"strategy{strategy}_history.json")

    if os.path.exists(save_path) and not force_retrain:
        print("=" * 60)
        print(f"STRATEGY {strategy} — loading from cache")
        print("=" * 60)
        print(f"  Path : {save_path}")
        model = tf.keras.models.load_model(save_path)

        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                hist_dict = json.load(f)
            class FakeHistory:
                def __init__(self, d): self.history = d
            history = FakeHistory(hist_dict)
            actual_epochs = len(hist_dict["loss"])
        else:
            history, actual_epochs = None, None

        print("  Status : loaded ✓")
        print("=" * 60)
        return model, history, actual_epochs

    print("=" * 60)
    print(f"STRATEGY {strategy} — training fresh")
    print(f"  {tag}")
    print("=" * 60)

    model = build_model(strategy)

    cb_list = [
        callbacks.EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss",
                                    factor=LR_PLATEAU_FACTOR,
                                    patience=LR_PLATEAU_PATIENCE,
                                    verbose=1, min_lr=1e-7),
        callbacks.ModelCheckpoint(filepath=save_path, monitor="val_loss",
                                  save_best_only=True, verbose=0),
    ]

    t0 = time.time()
    history = model.fit(
        train_dataset,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=cb_list,
        class_weight=class_weight,
        verbose=1,
    )
    elapsed_min = (time.time() - t0) / 60
    actual_epochs = len(history.history["loss"])

    model.save(save_path)
    hist_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(history_path, "w") as f:
        json.dump(hist_dict, f, indent=2)

    print("=" * 60)
    print(f"  Epochs run         : {actual_epochs}")
    print(f"  Training time      : {elapsed_min:.1f} min")
    print(f"  Saved model        : {save_path}")
    print(f"  Saved history      : {history_path}")
    print("=" * 60)

    return model, history, actual_epochs

##**Training Strategies**

### 11. Train Strategy 1 — Frozen Baseline

MobileNetV2 backbone fully frozen. Only the classification head is trained. No class weighting, no augmentation. Expected to be the fastest of the three to train.

In [ ]:
model_s1, hist_s1, epochs_s1 = train_strategy(
    strategy=1,
    train_dataset=train_ds_plain,
    class_weight=None,
    tag="Frozen Baseline (head only)"
)
gc.collect(); tf.keras.backend.clear_session()

### 12. Train Strategy 2 — Partial Fine-Tuning

Last 30 layers of MobileNetV2 plus the classification head are trainable. Lower-level features (edges, textures) remain frozen at ImageNet weights.

In [ ]:
model_s2, hist_s2, epochs_s2 = train_strategy(
    strategy=2,
    train_dataset=train_ds_plain,
    class_weight=None,
    tag="Partial Fine-Tuning (last 30 layers + head)"
)
gc.collect(); tf.keras.backend.clear_session()

### 13. Train Strategy 3 — Full Fine-Tuning + Class Weighting + Light Augmentation

All MobileNetV2 layers trainable. Class weights computed from the training set address the pneumonia class imbalance. Light augmentation is applied to the training set only.

In [ ]:
cw = compute_class_weight(class_weight="balanced",
                          classes=np.array([0, 1]),
                          y=df_train["label"].values)
class_weight_dict = {0: float(cw[0]), 1: float(cw[1])}

print("=" * 60)
print("CLASS WEIGHTS (Strategy 3)")
print("=" * 60)
print(f"  Class 0 (non-pneumonia): {class_weight_dict[0]:.4f}")
print(f"  Class 1 (pneumonia)    : {class_weight_dict[1]:.4f}")
print("=" * 60)

model_s3, hist_s3, epochs_s3 = train_strategy(
    strategy=3,
    train_dataset=train_ds_augment,
    class_weight=class_weight_dict,
    tag="Full FT + Class Weighting + Light Augmentation"
)
gc.collect(); tf.keras.backend.clear_session()

##**Evaluation, Visualization, and Export**

### 14. Evaluation Function

Computes Accuracy, Balanced Accuracy, AUROC, Precision, Recall (Sensitivity), Specificity, F1, and the confusion matrix using a fixed decision threshold of 0.5.

In [ ]:
def evaluate(model, dataset, y_true_array=None, name=""):
    y_prob = model.predict(dataset, verbose=0).ravel()

    if y_true_array is None:
        y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)
    else:
        y_true = y_true_array

    y_pred = (y_prob >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "Accuracy":         accuracy_score(y_true, y_pred),
        "BalancedAccuracy": (sens + spec) / 2,
        "AUROC":            roc_auc_score(y_true, y_prob),
        "Precision":        precision_score(y_true, y_pred, zero_division=0),
        "Recall":           sens,
        "Specificity":      spec,
        "F1":               f1_score(y_true, y_pred, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

    print("-" * 60)
    print(f"  {name}")
    print("-" * 60)
    for k in ["Accuracy", "BalancedAccuracy", "AUROC", "Precision", "Recall", "Specificity", "F1"]:
        print(f"    {k:<18}: {metrics[k]:.4f}")
    print(f"    Confusion matrix : TN={metrics['TN']}  FP={metrics['FP']}  FN={metrics['FN']}  TP={metrics['TP']}")
    return metrics

### 15. Evaluate All Strategies on Both Domains

Each model is evaluated twice: once on the NIH in-domain test set, once on the Guangzhou out-domain test set.

In [ ]:
results = {}

print("=" * 60)
print("EVALUATION — all strategies × both domains")
print("=" * 60)

for sid, model in [(1, model_s1), (2, model_s2), (3, model_s3)]:
    sname = {1: "S1_Frozen", 2: "S2_Partial", 3: "S3_FullFT_CW_Aug"}[sid]
    in_dom  = evaluate(model, test_ds, name=f"{sname}  |  NIH (in-domain)")
    out_dom = evaluate(model, gz_ds, y_true_array=gz_y, name=f"{sname}  |  Guangzhou (out-domain)")
    results[sname] = {"in_domain": in_dom, "out_domain": out_dom}

print("=" * 60)
print("  All evaluations complete ✓")
print("=" * 60)

### 16. Comparison Table and CSV Export

Build a single results table showing every metric for both domains and the performance drop (NIH − Guangzhou) per strategy. Save to CSV and JSON.

In [ ]:
metric_keys = ["Accuracy", "BalancedAccuracy", "AUROC", "Precision",
               "Recall", "Specificity", "F1"]

rows = []
for sname, r in results.items():
    row = {"Strategy": sname}
    for m in metric_keys:
        row[f"{m}_NIH"]  = r["in_domain"][m]
        row[f"{m}_GZ"]   = r["out_domain"][m]
        row[f"{m}_drop"] = r["in_domain"][m] - r["out_domain"][m]
    for k in ["TN", "FP", "FN", "TP"]:
        row[f"{k}_NIH"] = r["in_domain"][k]
        row[f"{k}_GZ"]  = r["out_domain"][k]
    rows.append(row)

results_df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "results_comparison.csv")
json_path = os.path.join(OUT_DIR, "results_full.json")
results_df.to_csv(csv_path, index=False)
with open(json_path, "w") as f:
    json.dump(results, f, indent=2, default=float)

print("=" * 90)
print(f"{'FINAL RESULTS — In-Domain (NIH) vs Out-Domain (Guangzhou)':^90}")
print("=" * 90)

for m in metric_keys:
    print(f"\n  {m}")
    print(f"  {'-'*70}")
    print(f"  {'Strategy':<22}{'NIH':>14}{'Guangzhou':>14}{'Drop':>14}")
    for _, r in results_df.iterrows():
        print(f"  {r['Strategy']:<22}{r[f'{m}_NIH']:>14.4f}{r[f'{m}_GZ']:>14.4f}{r[f'{m}_drop']:>+14.4f}")

print("\n" + "=" * 90)
print(f"  Saved CSV  : {csv_path}")
print(f"  Saved JSON : {json_path}")
print("=" * 90)

### 17. Treshold Calibration


In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

def find_optimal_threshold(model, val_dataset, method="f1"):
    y_prob = model.predict(val_dataset, verbose=0).ravel()
    y_true = np.concatenate([y.numpy() for _, y in val_dataset], axis=0)
    if method == "f1":
        prec, rec, thresh = precision_recall_curve(y_true, y_prob)
        f1_scores = 2 * (prec * rec) / (prec + rec + 1e-10)
        best_idx = np.argmax(f1_scores[:-1])
        best_thresh = float(thresh[best_idx])
        print(f"    Best F1 threshold: {best_thresh:.4f}  (val F1 = {f1_scores[best_idx]:.4f})")
    elif method == "youden":
        fpr, tpr, thresh = roc_curve(y_true, y_prob)
        j = tpr - fpr
        best_idx = np.argmax(j)
        best_thresh = float(thresh[best_idx])
        print(f"    Best Youden threshold: {best_thresh:.4f}")
    return best_thresh

def evaluate_calibrated(model, dataset, threshold, y_true_array=None, name=""):
    y_prob = model.predict(dataset, verbose=0).ravel()
    if y_true_array is None:
        y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)
    else:
        y_true = y_true_array
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics = {
        "Accuracy":         accuracy_score(y_true, y_pred),
        "BalancedAccuracy": (sens + spec) / 2,
        "AUROC":            roc_auc_score(y_true, y_prob),
        "Precision":        precision_score(y_true, y_pred, zero_division=0),
        "Recall":           sens,
        "Specificity":      spec,
        "F1":               f1_score(y_true, y_pred, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    print(f"  {name}  (thr = {threshold:.4f})")
    print(f"    Acc={metrics['Accuracy']:.4f}  BalAcc={metrics['BalancedAccuracy']:.4f}  "
          f"AUROC={metrics['AUROC']:.4f}  F1={metrics['F1']:.4f}")
    print(f"    Recall={metrics['Recall']:.4f}  Spec={metrics['Specificity']:.4f}  "
          f"CM: TN={tn} FP={fp} FN={fn} TP={tp}")
    return metrics

In [ ]:
print("=" * 70)
print("THRESHOLD CALIBRATION")
print("Using NIH validation set only")
print("=" * 70)

strategies = {
    "S1_Frozen": model_s1,
    "S2_Partial": model_s2,
    "S3_FullFT_CW_Aug": model_s3
}

results_calibrated = {}

for strategy_name, model in strategies.items():
    print("\n" + "-" * 70)
    print(f"Running calibration for: {strategy_name}")
    print("-" * 70)

    # Find the best threshold using NIH validation set only
    best_threshold = find_optimal_threshold(
        model=model,
        val_dataset=val_ds,
        method="f1"
    )

    # Evaluate on NIH in-domain test set
    nih_result = evaluate_calibrated(
        model=model,
        dataset=test_ds,
        threshold=best_threshold,
        name=f"{strategy_name} | NIH in-domain"
    )

    # Evaluate on Guangzhou external test set
    gz_result = evaluate_calibrated(
        model=model,
        dataset=gz_ds,
        threshold=best_threshold,
        y_true_array=gz_y,
        name=f"{strategy_name} | Guangzhou out-domain"
    )

    # Store results
    results_calibrated[strategy_name] = {
        "threshold": best_threshold,
        "in_domain": nih_result,
        "out_domain": gz_result
    }

print("\n" + "=" * 70)
print("THRESHOLD CALIBRATION COMPLETED")
print("=" * 70)

In [ ]:
rows_cal = []
for sname, r in results_calibrated.items():
    row = {"Strategy": sname, "Threshold": round(r["threshold"], 4)}
    for m in metric_keys:
        row[f"{m}_NIH"]  = round(r["in_domain"][m], 4)
        row[f"{m}_GZ"]   = round(r["out_domain"][m], 4)
        row[f"{m}_drop"] = round(r["in_domain"][m] - r["out_domain"][m], 4)
    for k in ["TN", "FP", "FN", "TP"]:
        row[f"{k}_NIH"] = r["in_domain"][k]
        row[f"{k}_GZ"]  = r["out_domain"][k]
    rows_cal.append(row)

df_calibrated = pd.DataFrame(rows_cal)
csv_cal = os.path.join(OUT_DIR, "results_calibrated.csv")
df_calibrated.to_csv(csv_cal, index=False)
print(f"\n  Saved: {csv_cal}")

In [ ]:
print("\n" + "=" * 100)
print(f"{'COMPARISON: DEFAULT (0.5) vs CALIBRATED':^100}")
print("=" * 100)
for m in ["AUROC", "F1", "Recall", "Specificity"]:
    print(f"\n  {m}")
    print(f"  {'Strategy':<22}{'NIH(0.5)':>11}{'NIH(cal)':>11}{'GZ(0.5)':>11}{'GZ(cal)':>11}{'Drop(cal)':>13}")
    for sname in results_calibrated:
        r0 = results[sname]
        rc = results_calibrated[sname]
        print(f"  {sname:<22}"
              f"{r0['in_domain'][m]:>11.4f}"
              f"{rc['in_domain'][m]:>11.4f}"
              f"{r0['out_domain'][m]:>11.4f}"
              f"{rc['out_domain'][m]:>11.4f}"
              f"{rc['in_domain'][m] - rc['out_domain'][m]:>+13.4f}")
print("=" * 100)

### 18. Visualizations



Calibrated confusion matrices

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 13))

for i, sname in enumerate(strat_order):
    for j, (dkey, dlabel) in enumerate([("in_domain", "NIH (in-domain)"),
                                         ("out_domain", "Guangzhou (out-domain)")]):
        m = results_calibrated[sname][dkey]
        thr = results_calibrated[sname]["threshold"]
        cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
        ax = axes[i, j]
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred 0\nnon-pneumonia", "Pred 1\npneumonia"])
        ax.set_yticklabels(["True 0", "True 1"])
        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(cm[r, c]), ha="center", va="center",
                        color="white" if cm[r, c] > cm.max()/2 else "black",
                        fontsize=14, fontweight="bold")
        ax.set_title(f"{strat_nice[sname]}\n{dlabel}  (thr = {thr:.4f})", fontsize=11)

plt.suptitle("Confusion Matrices — Calibrated Threshold",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
p = os.path.join(OUT_DIR, "cm_calibrated_threshold.png")
plt.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {p}")

Cross-domain bar chart

In [ ]:
strat_keys = list(results_calibrated.keys())
display_labels = ["S1\nFrozen", "S2\nPartial FT", "S3\nFull FT+CW+Aug"]
panel_metrics = ["AUROC", "F1", "Recall", "Specificity"]
panel_titles  = {"AUROC": "AUROC", "F1": "F1 Score",
                 "Recall": "Recall (Sensitivity)", "Specificity": "Specificity"}

nih_v = {m: [results_calibrated[s]["in_domain"][m]  for s in strat_keys] for m in panel_metrics}
gz_v  = {m: [results_calibrated[s]["out_domain"][m] for s in strat_keys] for m in panel_metrics}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
x = np.arange(len(strat_keys)); w = 0.33

for ax, m in zip(axes.flat, panel_metrics):
    b1 = ax.bar(x - w/2, nih_v[m], w, label="NIH (in-domain)",
                color="#1E3A8A", edgecolor="white", linewidth=0.5)
    b2 = ax.bar(x + w/2, gz_v[m],  w, label="Guangzhou (out-domain)",
                color="#93C5FD", edgecolor="white", linewidth=0.5)
    for bars in [b1, b2]:
        for b in bars:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.015, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=8.5, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(display_labels, fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.set_title(panel_titles[m], fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.2); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=2, frameon=False,
           fontsize=11, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Cross-Hospital Performance (Calibrated Threshold)",
             fontsize=14, fontweight="bold", y=1.06)
plt.tight_layout(rect=[0, 0, 1, 0.97])
p = os.path.join(OUT_DIR, "fig_cross_domain_comparison.png")
plt.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"  Saved: {p}")


Performance Drop Chart

In [ ]:
drop_metrics = ["AUROC", "F1", "Recall", "Specificity"]
drop_colors  = ["#1E3A8A", "#3B82F6", "#60A5FA", "#93C5FD"]

drops = {}
for m in drop_metrics:
    drops[m] = [results_calibrated[s]["in_domain"][m] - results_calibrated[s]["out_domain"][m]
                for s in strat_keys]

fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(strat_keys))
n_m = len(drop_metrics); grp_w = 0.75; bar_w = grp_w / n_m

for i, (m, c) in enumerate(zip(drop_metrics, drop_colors)):
    offset = (i - (n_m - 1) / 2) * bar_w
    bars = ax.bar(x + offset, drops[m], bar_w, label=m,
                  color=c, edgecolor="white", linewidth=0.5)
    for b in bars:
        h = b.get_height()
        va = "bottom" if h >= 0 else "top"
        ax.text(b.get_x() + b.get_width()/2, h + (0.012 if h >= 0 else -0.012),
                f"{h:+.3f}", ha="center", va=va, fontsize=8)

ax.axhline(0, color="black", linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(["S1 Frozen", "S2 Partial FT", "S3 Full FT+CW+Aug"],
                    fontsize=10, fontweight="bold")
ax.set_ylabel("Performance Drop (NIH − Guangzhou)")
ax.set_title("Performance Drop Per Strategy\n(negative = better on Guangzhou)",
             fontsize=13, fontweight="bold", pad=14)
ax.legend(loc="lower left", frameon=True, fancybox=False, edgecolor="#ddd", fontsize=10, ncol=4)
ax.grid(axis="y", alpha=0.2); ax.set_axisbelow(True)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ylim = ax.get_ylim()
ax.axhspan(ylim[0], 0, alpha=0.05, color="#1E3A8A")
ax.set_ylim(ylim)
ax.text(0.99, 0.03, "better on Guangzhou", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color="#1E3A8A", fontstyle="italic", alpha=0.6)
ax.text(0.99, 0.97, "better on NIH", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, color="#7C2D12", fontstyle="italic", alpha=0.6)

plt.tight_layout()
p = os.path.join(OUT_DIR, "fig_performance_drop.png")
plt.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"  Saved: {p}")

Training History

In [ ]:
strat_hist_names = {
    "S1_Frozen":        "Strategy 1: Frozen Baseline",
    "S2_Partial":       "Strategy 2: Partial Fine-Tuning",
    "S3_FullFT_CW_Aug": "Strategy 3: Full FT + CW + Aug",
}

hist_data = {}
for sid, sname in [(1, "S1_Frozen"), (2, "S2_Partial"), (3, "S3_FullFT_CW_Aug")]:
    hp = os.path.join(OUT_DIR, f"strategy{sid}_history.json")
    if os.path.exists(hp):
        with open(hp) as f:
            hist_data[sname] = json.load(f)

fig, axes = plt.subplots(3, 2, figsize=(13, 11))

for i, (sname, h) in enumerate(hist_data.items()):
    epochs = range(1, len(h["loss"]) + 1)

    axes[i, 0].plot(epochs, h["loss"], color="#1E3A8A", linewidth=2, label="train")
    axes[i, 0].plot(epochs, h["val_loss"], color="#60A5FA", linewidth=2, label="val", linestyle="--")
    axes[i, 0].fill_between(epochs, h["loss"], alpha=0.08, color="#1E3A8A")
    axes[i, 0].set_title(f"{strat_hist_names[sname]} — Loss", fontweight="bold")
    axes[i, 0].set_xlabel("Epoch"); axes[i, 0].set_ylabel("Loss")
    axes[i, 0].legend(frameon=False); axes[i, 0].grid(alpha=0.2)
    axes[i, 0].spines["top"].set_visible(False); axes[i, 0].spines["right"].set_visible(False)

    axes[i, 1].plot(epochs, h["auc"], color="#1E3A8A", linewidth=2, label="train")
    axes[i, 1].plot(epochs, h["val_auc"], color="#60A5FA", linewidth=2, label="val", linestyle="--")
    axes[i, 1].fill_between(epochs, h["auc"], alpha=0.08, color="#1E3A8A")
    axes[i, 1].set_title(f"{strat_hist_names[sname]} — AUC", fontweight="bold")
    axes[i, 1].set_xlabel("Epoch"); axes[i, 1].set_ylabel("AUC")
    axes[i, 1].legend(frameon=False); axes[i, 1].grid(alpha=0.2)
    axes[i, 1].spines["top"].set_visible(False); axes[i, 1].spines["right"].set_visible(False)

plt.tight_layout()
p = os.path.join(OUT_DIR, "fig_training_history.png")
plt.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"  Saved: {p}")

Summary Dashboard

In [ ]:
short_labels = ["S1 Frozen", "S2 Partial", "S3 Full+CW+Aug"]
bar_colors = ["#1E3A8A", "#3B82F6", "#93C5FD"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel 1: Thresholds
thresholds = [results_calibrated[s]["threshold"] for s in strat_keys]
bars1 = axes[0].barh(short_labels, thresholds, color=bar_colors, edgecolor="white", height=0.5)
axes[0].axvline(0.5, color="#DC2626", linewidth=1.5, linestyle="--", alpha=0.6, label="default 0.5")
for b, v in zip(bars1, thresholds):
    axes[0].text(v + 0.01, b.get_y() + b.get_height()/2,
                 f"{v:.4f}", va="center", fontsize=10, fontweight="bold")
axes[0].set_xlim(0, 0.6); axes[0].set_title("Calibrated Threshold", fontweight="bold")
axes[0].legend(frameon=False, fontsize=9)

# Panel 2: GZ AUROC
gz_auroc = [results_calibrated[s]["out_domain"]["AUROC"] for s in strat_keys]
bars2 = axes[1].barh(short_labels, gz_auroc, color=bar_colors, edgecolor="white", height=0.5)
axes[1].axvline(0.5, color="#DC2626", linewidth=1.5, linestyle="--", alpha=0.6, label="random")
for b, v in zip(bars2, gz_auroc):
    axes[1].text(v + 0.01, b.get_y() + b.get_height()/2,
                 f"{v:.3f}", va="center", fontsize=10, fontweight="bold")
axes[1].set_xlim(0, 1.0); axes[1].set_title("Guangzhou AUROC", fontweight="bold")
axes[1].legend(frameon=False, fontsize=9)

# Panel 3: GZ F1 + Recall
xi = np.arange(len(strat_keys)); bw = 0.3
gz_f1  = [results_calibrated[s]["out_domain"]["F1"]    for s in strat_keys]
gz_rec = [results_calibrated[s]["out_domain"]["Recall"] for s in strat_keys]
axes[2].barh(xi - bw/2, gz_f1,  bw, label="F1",     color="#1E3A8A", edgecolor="white")
axes[2].barh(xi + bw/2, gz_rec, bw, label="Recall",  color="#93C5FD", edgecolor="white")
for xii, (fv, rv) in enumerate(zip(gz_f1, gz_rec)):
    axes[2].text(fv + 0.01, xii - bw/2, f"{fv:.3f}", va="center", fontsize=9)
    axes[2].text(rv + 0.01, xii + bw/2, f"{rv:.3f}", va="center", fontsize=9)
axes[2].set_yticks(xi); axes[2].set_yticklabels(short_labels)
axes[2].set_xlim(0, 1.0); axes[2].set_title("Guangzhou F1 + Recall", fontweight="bold")
axes[2].legend(frameon=False, fontsize=9)

for ax in axes:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(axis="x", alpha=0.2)

plt.suptitle("Summary Dashboard", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
p = os.path.join(OUT_DIR, "fig_summary_dashboard.png")
plt.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"  Saved: {p}")

## 19. Interpretation Guide

How to read the final results table for the paper's discussion section.

In [ ]:
print("=" * 78)
print(f"{'INTERPRETATION GUIDE':^78}")
print("=" * 78)
print("""
  TWO RESULT SETS:
    1. Default threshold (0.5) — pre-specified, no tuning
    2. Calibrated threshold — tuned per strategy on validation F1

  WHY CALIBRATION MATTERS:
    At threshold 0.5, S1 and S2 predicted 0% pneumonia (all outputs < 0.5
    due to class imbalance). Calibration found thresholds of ~0.08-0.10
    which unlocked their discriminative signal.

  PRIMARY METRICS (use calibrated):
    AUROC        prevalence-invariant, best for cross-domain comparison
    F1           balanced precision + recall
    Recall       clinically critical (catching pneumonia)

  PERFORMANCE DROP:
    Drop = NIH - Guangzhou
    Negative drop = BETTER on external domain (good for robustness)

  KEY FINDINGS:
    S1 Frozen     best GZ AUROC (0.777), negative AUROC drop
    S2 Partial    best GZ F1 (0.722) and Recall (0.674)
    S3 Full+CW    weakest on both domains, overfit to NIH

  REPORT IN PAPER:
    - S2 as most robust strategy (highest external F1 + Recall)
    - S1 as best ranker (highest external AUROC)
    - S3 as cautionary finding (overfitting under limited data)
    - Threshold calibration as essential methodological step
""")

# Print actual training epochs
print("  ACTUAL TRAINING EPOCHS (EarlyStopping):")
for sid in [1, 2, 3]:
    hp = os.path.join(OUT_DIR, f"strategy{sid}_history.json")
    if os.path.exists(hp):
        with open(hp) as f:
            n_ep = len(json.load(f)["loss"])
        print(f"    Strategy {sid}: {n_ep} epochs")
print("=" * 78)